In [1]:
# 1. Check GPU allocation
!nvidia-smi

# 2. Mount Google Drive to persist data and access the repository
from google.colab import drive
drive.mount('/content/drive')

Sun Aug  2 04:13:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Di chuyển vào thư mục chứa custom Ultralytics đã upload
# Hãy thay thế đường dẫn bên dưới bằng vị trí thực tế của bạn
%cd /content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
# 2. Gỡ bản gốc để tránh xung đột
!pip uninstall -y ultralytics

# 3. Cài đặt bản custom (lệnh này sẽ cài đặt thư viện vào hệ thống Colab)
!pip install -e .

/content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
Found existing installation: ultralytics 8.3.231
Uninstalling ultralytics-8.3.231:
  Successfully uninstalled ultralytics-8.3.231
Obtaining file:///content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.231-0.editable-py3-none-any.whl size=11709 sha256=2d78dcbfc9f1b5aab4cb673eecfc80824a55b9ad6c6eed7d7f02273383f2294b
  Stored in directory: /tmp/pip-ephem-wheel-cache-trqdfzzh/wheels/6d/54/88/b24c9a513438bdb6003f090cbfc70bfc9b5fc534848fa36efd
Successfully built ultralytics


In [3]:
# Di chuyển về thư mục gốc của dự án
%cd /content/drive/MyDrive/ppe_cv/ppe_v2

# Kiểm tra lại xem đã đứng đúng thư mục chưa (sẽ thấy thư mục scripts, file data.yaml...)
!ls

/content/drive/MyDrive/ppe_cv/ppe_v2
assets	     data.yaml			requirements.txt  yolo11n.pt
bench.ipynb  demo.py			runs		  yolov5_au.ipynb
config.py    experiments_dagshub.ipynb	scripts		  yolov8_baseline.ipynb
configs      README.md			weights		  yolov8_prune.ipynb


In [4]:
!pip install mlflow

In [5]:
import os
import sys

# 1. Khai báo đường dẫn tới thư mục repo custom Ultralytics
custom_repo_path = "/content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices"
if custom_repo_path not in sys.path:
    sys.path.insert(0, custom_repo_path)

# 2. Di chuyển vào thư mục dự án ppe_v2
%cd /content/drive/MyDrive/ppe_cv/ppe_v2

# 3. Cho phép MLflow File Store
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

print("Đã thiết lập môi trường sẵn sàng cho Giai đoạn 3!")

/content/drive/MyDrive/ppe_cv/ppe_v2
Đã thiết lập môi trường sẵn sàng cho Giai đoạn 3!


In [6]:
import torch
torch.cuda.empty_cache()

In [7]:
# Thực thi huấn luyện phục hồi
# Lưu ý: batch size được giữ ở mức 16 như các cấu hình trước của bạn
!python scripts/train.py \
    --mode finetune \
    --weights /content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.30.pt \
    --data data.yaml \
    --epochs 100 \
    --batch 32 \
    --device 0

New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.231 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe_cv/ppe_v2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, finetune=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.30.pt, moment

In [9]:
# Đánh giá chỉ số mAP của mô hình tốt nhất sau Giai đoạn 3
!python scripts/val.py \
    --weight runs/train-finetune/weights/best.pt \
    --data data.yaml \
    --split val \
    --batch 16 \
    --imgsz 640

Average Latency per image: 16.60 ms
Approx FPS: 60.26
Ultralytics 8.3.231 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 14,044,710 parameters, 0 gradients, 57.8 GFLOPs
val: Fast image access ✅ (ping: 11.4±24.4 ms, read: 7.7±6.8 MB/s, size: 58.2 KB)
val: Scanning /content/drive/MyDrive/ppe_cv/train_data/valid/labels.cache... 229 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229 2.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.8it/s 8.5s
                   all        229       2019      0.753      0.645      0.696      0.358
                 boots         93        201      0.605      0.542      0.553      0.237
                gloves         78        157      0.679      0.405      0.478      0.163
                helmet        130        337      0.825      0.804       0.87      0.527
              no_boots         32         65      0.674     

In [12]:
import mlflow
import os
import getpass

DAGSHUB_USER = input("Nhập DagsHub Username: ")
DAGSHUB_TOKEN = getpass.getpass("Nhập DagsHub Token: ")
DAGSHUB_REPO = input("Nhập tên DagsHub Repo (VD: ppe-project): ")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow"

Nhập DagsHub Username: TQanh23
Nhập DagsHub Token: ··········
Nhập tên DagsHub Repo (VD: ppe-project): ppe_v2


In [27]:
mlflow.set_experiment("YOLOv8_Finetuning")

with mlflow.start_run(run_name="Phase3_Finetune_r0.30"):
    # Log tham số huấn luyện
    mlflow.log_params({
        "mode": "finetune",
        "epochs": 100,
        "batch_size": 32,
        "lr0": 0.001,
        "source_weights": "/content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.30.pt"
    })

    # Đường dẫn tới thư mục gốc của phiên huấn luyện
    run_dir = "/content/drive/MyDrive/ppe_cv/ppe_v2/runs/train-finetune"

    if os.path.exists(run_dir):
        # Lệnh này sẽ upload TOÀN BỘ file và thư mục con bên trong train-finetune4
        mlflow.log_artifacts(local_dir=run_dir, artifact_path="training_results")

        print(f"✅ Đã log toàn bộ artifacts lên MLflow thành công!")
        print(f"📁 Các file được upload bao gồm: weights (best.pt, last.pt), biểu đồ metrics, confusion matrix, kết quả CSV...")
    else:
        print(f"❌ Không tìm thấy thư mục {run_dir}. Hãy kiểm tra lại đường dẫn.")

✅ Đã log toàn bộ artifacts lên MLflow thành công!
📁 Các file được upload bao gồm: weights (best.pt, last.pt), biểu đồ metrics, confusion matrix, kết quả CSV...
🏃 View run Phase3_Finetune_r0.30 at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/10/runs/0eab99334ac142a39c0244305f045520
🧪 View experiment at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/10


In [22]:
!ls /content/drive/MyDrive/ppe_cv/ppe_v2/runs/

train-finetune	train-sparsity	train-sparsity2  train-sparsity3
